# Dərs 18 — Advanced Spark Transformations


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, ArrayType
)

spark = (
    SparkSession.builder
    .appName("Lesson22-Advanced-Transformations")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "matrix")
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)
spark.version

'3.5.2'

In [33]:
customers = spark.read.option("header", True).option("inferSchema", True).csv("s3a://matrix/raw/customers.csv")
card_trn = spark.read.option("header", True).option("inferSchema", True).csv("s3a://matrix/raw/card_trn.csv")
trn_items = spark.read.json("s3a://matrix/raw/trn_items.json")

customers.printSchema()
card_trn.printSchema()
trn_items.printSchema()   

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- balance: double (nullable = true)
 |-- vip_status: string (nullable = true)

root
 |-- trn_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- trn_date: date (nullable = true)
 |-- merchant: string (nullable = true)

root
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- product_name: string (nullable = true)
 |    |    |-- qty: long (nullable = true)
 |    |    |-- unit_price: double (nullable = true)
 |-- trn_id: string (nullable = true)



## 1. Joins (Slayd 2–3)


In [34]:
# INNER — yalnız tranzaksiyası olan müştərilər
customers_with_trn = customers.join(card_trn, "customer_id")
print("Inner join sətir sayı:", customers_with_trn.count())

# LEFT ANTI — heç tranzaksiyası olmayan müştərilər (C018, C019, C020 gözlənilir)
no_trn_customers = customers.join(card_trn, "customer_id", "left_anti")
print("Tranzaksiyası olmayan müştəri sayı:", no_trn_customers.count())
no_trn_customers.select("customer_id", "customer_name").show()

Inner join sətir sayı: 45
Tranzaksiyası olmayan müştəri sayı: 3
+-----------+-----------------+
|customer_id|    customer_name|
+-----------+-----------------+
|       C018|     Aysel Aliyev|
|       C020|     Leyla Rzayev|
|       C019|Nargiz Ismayilova|
+-----------+-----------------+



`customers.join(card_trn, "customer_id", "left")` ilə LEFT join et. Nəticədə `no_trn_customers`-dəki müştərilər üçün `amount` sütununda nə görünür? Niyə?


In [6]:

left_joined = customers.join(card_trn, "customer_id", "left")
left_joined.filter(F.col("customer_id").isin("C018", "C019", "C020")).show()

+-----------+-----------------+---+-------+----------+------+------+--------+--------+
|customer_id|    customer_name|age|balance|vip_status|trn_id|amount|trn_date|merchant|
+-----------+-----------------+---+-------+----------+------+------+--------+--------+
|       C018|     Aysel Aliyev| 27|4073.37|         N|  NULL|  NULL|    NULL|    NULL|
|       C019|Nargiz Ismayilova| 22| 897.11|         N|  NULL|  NULL|    NULL|    NULL|
|       C020|     Leyla Rzayev| 38|4400.35|         N|  NULL|  NULL|    NULL|    NULL|
+-----------+-----------------+---+-------+----------+------+------+--------+--------+



In [35]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")


**Tapşırıq 1.2 (Broadcast Join):** `customers` DataFrame-i kiçikdir (20 sətir). `broadcast()` funksiyasını import edib, inner join-i broadcast join kimi yenidən yaz. `.explain()` çıxışında `BroadcastHashJoin` sözünü tap (adi join-də isə `SortMergeJoin` görəcəksən).


In [36]:
from pyspark.sql.functions import broadcast


customers_with_trn.explain()

customers_with_trn_bc = broadcast(customers).join(card_trn, "customer_id", "inner")
customers_with_trn_bc.explain()

== Physical Plan ==
*(5) Project [customer_id#908, customer_name#909, age#910, balance#911, vip_status#912, trn_id#935, amount#937, trn_date#938, merchant#939]
+- *(5) SortMergeJoin [customer_id#908], [customer_id#936], Inner
   :- *(2) Sort [customer_id#908 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(customer_id#908, 200), ENSURE_REQUIREMENTS, [plan_id=2108]
   :     +- *(1) Filter isnotnull(customer_id#908)
   :        +- FileScan csv [customer_id#908,customer_name#909,age#910,balance#911,vip_status#912] Batched: false, DataFilters: [isnotnull(customer_id#908)], Format: CSV, Location: InMemoryFileIndex(1 paths)[s3a://matrix/raw/customers.csv], PartitionFilters: [], PushedFilters: [IsNotNull(customer_id)], ReadSchema: struct<customer_id:string,customer_name:string,age:int,balance:double,vip_status:string>
   +- *(4) Sort [customer_id#936 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(customer_id#936, 200), ENSURE_REQUIREMENTS, [plan_id=2116]
         +

## 2. Window Functions 


In [19]:
w = Window.partitionBy("customer_id").orderBy(F.desc("amount"))

ranked = (
    customers_with_trn
    .withColumn("row_num", F.row_number().over(w))
    .withColumn("rnk", F.rank().over(w))
    .withColumn("dense_rnk", F.dense_rank().over(w))
)
ranked.select("customer_id", "amount", "row_num", "rnk", "dense_rnk").show(15)

+-----------+------+-------+---+---------+
|customer_id|amount|row_num|rnk|dense_rnk|
+-----------+------+-------+---+---------+
|       C006|166.53|      1|  1|        1|
|       C010| 251.5|      1|  1|        1|
|       C007| 330.7|      1|  1|        1|
|       C007|112.19|      2|  2|        2|
|       C012|407.91|      1|  1|        1|
|       C012| 233.8|      2|  2|        2|
|       C003|325.72|      1|  1|        1|
|       C003|290.23|      2|  2|        2|
|       C003| 270.4|      3|  3|        3|
|       C003|270.17|      4|  4|        4|
|       C003|198.47|      5|  5|        5|
|       C003|116.88|      6|  6|        6|
|       C015|192.74|      1|  1|        1|
|       C015| 89.64|      2|  2|        2|
|       C004|394.75|      1|  1|        1|
+-----------+------+-------+---+---------+
only showing top 15 rows



In [ ]:
top2_per_customer = ranked.filter(F.col("row_num") <= 2)
top2_per_customer.select("customer_id", "amount", "trn_date", "merchant").orderBy("customer_id", "row_num").show(20)

+-----------+------+----------+------------+
|customer_id|amount|  trn_date|    merchant|
+-----------+------+----------+------------+
|       C001|265.14|2026-07-11|     Ecoteks|
|       C001| 67.14|2026-03-09|       SOCAR|
|       C002|448.81|2026-02-26|        Wolt|
|       C002|224.61|2026-05-11|       Bravo|
|       C003|325.72|2026-06-24|       SOCAR|
|       C003|290.23|2026-07-16|Kontakt Home|
|       C004|394.75|2026-07-04|       SOCAR|
|       C005|429.31|2026-05-19|       Bravo|
|       C005|283.26|2026-08-11|        Bolt|
|       C006|166.53|2026-08-26|       SOCAR|
|       C007| 330.7|2026-01-22|Kontakt Home|
|       C007|112.19|2026-01-15|Kontakt Home|
|       C008|418.37|2026-02-24|        Bolt|
|       C008|405.33|2026-04-23|       Umico|
|       C009|104.99|2026-03-05| Araz Market|
|       C010| 251.5|2026-05-31|        Bolt|
|       C011|319.99|2026-01-19|       Bravo|
|       C012|407.91|2026-01-18| Araz Market|
|       C012| 233.8|2026-01-19|       SOCAR|
|       C0

## 3. Union 


In [37]:
jan_trn = card_trn.filter(F.month("trn_date") <= 4)
rest_trn = card_trn.filter(F.month("trn_date") > 4)

combined = jan_trn.select("customer_id", "amount", "trn_id", "trn_date", "merchant") \
    .unionByName(rest_trn.select("trn_id", "customer_id", "merchant", "amount", "trn_date"))



 Eyni iki DataFrame-i `union()` (unionByName yox) ilə birləşdirməyə çalış — sütun sırası fərqli olduğu üçün nəticə niyə səhvdir? `show()` ilə göstər.


## 4. Pivot / Unpivot (Slayd 7)


In [23]:
pivoted = (
    card_trn
    .withColumn("month", F.month("trn_date"))
    .groupBy("customer_id")
    .pivot("month")
    .agg(F.sum("amount"))
)
pivoted.show(10)

+-----------+------+------+------+-----------------+------+------+------+------+
|customer_id|     1|     2|     3|                4|     5|     6|     7|     8|
+-----------+------+------+------+-----------------+------+------+------+------+
|       C006|  NULL|  NULL|  NULL|             NULL|  NULL|  NULL|  NULL|166.53|
|       C010|  NULL|  NULL|  NULL|             NULL| 251.5|  NULL|  NULL|  NULL|
|       C007|442.89|  NULL|  NULL|             NULL|  NULL|  NULL|  NULL|  NULL|
|       C012|641.71|  NULL|  NULL|             NULL|  NULL|  NULL|  NULL|  NULL|
|       C003|  NULL|  NULL|198.47|540.5699999999999|116.88|325.72|290.23|  NULL|
|       C015|  NULL|282.38|  NULL|             NULL|  NULL|  NULL|  NULL|  NULL|
|       C004|  NULL|  NULL|  NULL|             NULL|  NULL|  NULL|394.75|  NULL|
|       C011|319.99|  NULL|  NULL|             NULL|  NULL|  NULL|  NULL|  NULL|
|       C014|  NULL|  NULL|  NULL|           329.06|  NULL|  NULL|  NULL|  NULL|
|       C017|  NULL|  NULL| 

Yuxarıdakı pivot cədvəlində NULL-lar var (o ay heç tranzaksiya olmayanda). `.na.fill(0)` ilə bunları 0-a çevir.


In [ ]:
pivoted_filled = pivoted.na.fill(0)
pivoted_filled.show(10)

+-----------+------+------+------+-----------------+------+------+------+------+
|customer_id|     1|     2|     3|                4|     5|     6|     7|     8|
+-----------+------+------+------+-----------------+------+------+------+------+
|       C006|   0.0|   0.0|   0.0|              0.0|   0.0|   0.0|   0.0|166.53|
|       C010|   0.0|   0.0|   0.0|              0.0| 251.5|   0.0|   0.0|   0.0|
|       C007|442.89|   0.0|   0.0|              0.0|   0.0|   0.0|   0.0|   0.0|
|       C012|641.71|   0.0|   0.0|              0.0|   0.0|   0.0|   0.0|   0.0|
|       C003|   0.0|   0.0|198.47|540.5699999999999|116.88|325.72|290.23|   0.0|
|       C015|   0.0|282.38|   0.0|              0.0|   0.0|   0.0|   0.0|   0.0|
|       C004|   0.0|   0.0|   0.0|              0.0|   0.0|   0.0|394.75|   0.0|
|       C011|319.99|   0.0|   0.0|              0.0|   0.0|   0.0|   0.0|   0.0|
|       C014|   0.0|   0.0|   0.0|           329.06|   0.0|   0.0|   0.0|   0.0|
|       C017|   0.0|   0.0| 

In [25]:
trn_items.show()

+--------------------+------+
|               items|trn_id|
+--------------------+------+
|[{Cheese, 1, 5.5}...| T0001|
|[{Snack Bar, 1, 1...| T0002|
|[{Charger, 3, 12....| T0003|
|  [{Coffee, 3, 8.9}]| T0004|
|[{Snack Bar, 3, 1...| T0005|
|[{Bread, 1, 1.2},...| T0006|
|[{Snack Bar, 2, 1...| T0007|
|[{Milk, 3, 2.1}, ...| T0008|
|[{Pen, 3, 0.8}, {...| T0009|
|[{Snack Bar, 1, 1...| T0010|
|[{Headphones, 3, ...| T0011|
|[{Headphones, 2, ...| T0012|
|[{Shampoo, 1, 4.7...| T0013|
|[{Water, 3, 0.9},...| T0014|
|    [{Milk, 3, 2.1}]| T0015|
|[{Cheese, 2, 5.5}...| T0016|
|[{Pen, 2, 0.8}, {...| T0017|
|    [{Milk, 2, 2.1}]| T0018|
|[{Shampoo, 3, 4.7...| T0019|
|[{Notebook, 2, 3....| T0020|
+--------------------+------+
only showing top 20 rows



## 5. Explode 


In [26]:
exploded_items = trn_items.select("trn_id", F.explode("items").alias("item"))
exploded_items.printSchema()
exploded_items.select("trn_id", "item.product_name", "item.qty", "item.unit_price").show(10)

root
 |-- trn_id: string (nullable = true)
 |-- item: struct (nullable = true)
 |    |-- product_name: string (nullable = true)
 |    |-- qty: long (nullable = true)
 |    |-- unit_price: double (nullable = true)

+------+------------+---+----------+
|trn_id|product_name|qty|unit_price|
+------+------------+---+----------+
| T0001|      Cheese|  1|       5.5|
| T0001|     Charger|  3|      12.5|
| T0001|       Bread|  1|       1.2|
| T0002|   Snack Bar|  1|       1.5|
| T0002|       Bread|  1|       1.2|
| T0002|       Water|  1|       0.9|
| T0003|     Charger|  3|      12.5|
| T0003|        Milk|  2|       2.1|
| T0003|      Coffee|  1|       8.9|
| T0003|   Detergent|  3|       6.4|
+------+------------+---+----------+
only showing top 10 rows



## 6. Arrays & Structs 


In [ ]:
# Neçə məhsul növü alınıb — array funksiyaları
trn_items_enriched = (
    trn_items
    .withColumn("n_items", F.size("items"))
    .withColumn("product_names", F.col("items.product_name"))   
)
trn_items_enriched.select("trn_id", "n_items", "product_names").show(10, truncate=False)

+------+-------+----------------------------------+
|trn_id|n_items|product_names                     |
+------+-------+----------------------------------+
|T0001 |3      |[Cheese, Charger, Bread]          |
|T0002 |3      |[Snack Bar, Bread, Water]         |
|T0003 |4      |[Charger, Milk, Coffee, Detergent]|
|T0004 |1      |[Coffee]                          |
|T0005 |1      |[Snack Bar]                       |
|T0006 |4      |[Bread, Coffee, Shampoo, Water]   |
|T0007 |1      |[Snack Bar]                       |
|T0008 |2      |[Milk, Snack Bar]                 |
|T0009 |2      |[Pen, Milk]                       |
|T0010 |1      |[Snack Bar]                       |
+------+-------+----------------------------------+
only showing top 10 rows



`F.array_contains(F.col("items.product_name"), "Coffee")` ilə tərkibində "Coffee" olan tranzaksiyaları tap.


In [ ]:
coffee_transactions = trn_items_enriched.filter( F.array_contains(F.col("items.product_name"), "Coffee"))

coffee_transactions.select("trn_id", "product_names").show(truncate=False)


exploded_items.filter(F.col("item.product_name") == "Coffee") \
    .select("trn_id").distinct()

+------+----------------------------------+
|trn_id|product_names                     |
+------+----------------------------------+
|T0003 |[Charger, Milk, Coffee, Detergent]|
|T0004 |[Coffee]                          |
|T0006 |[Bread, Coffee, Shampoo, Water]   |
|T0013 |[Shampoo, Coffee]                 |
|T0025 |[Pen, Detergent, Charger, Coffee] |
|T0026 |[Detergent, Coffee]               |
|T0031 |[Milk, Coffee]                    |
|T0042 |[Cheese, Pen, Coffee]             |
|T0043 |[Detergent, Coffee, Snack Bar]    |
|T0044 |[Coffee, Water]                   |
+------+----------------------------------+



## 7. Nested Data 

In [ ]:
full_picture = (
    customers
    .join(card_trn, "customer_id", "inner")
    .join(trn_items, "trn_id", "inner")
    .select(
        "customer_id", "customer_name", "trn_id", "trn_date",
        F.explode("items").alias("item")
    )
    .select(
        "customer_id", "customer_name", "trn_id", "trn_date",
        "item.product_name", "item.qty", "item.unit_price"
    )
)
full_picture.show(10, truncate=False)